# Data Ingestion

So we separate ingestion from querying. One process writes the data to a persistent search index. Another process reads from it. The two run independently and only share the index between them.

The index survives restarts, so we ingest once and query as often as we like. This is the ingestion step from data engineering. We move data from its source into a target system the application can use.

You can use any persistent search backend for this, such as Elasticsearch, OpenSearch, or Qdrant. In this module, we use sqlitesearch, a lightweight search library backed by SQLite FTS5. It has the same API as minsearch, so it's a drop-in replacement that happens to be persistent.

I picked SQLite because it asks nothing of you. It ships with Python, so you don't add any dependency, and it has FTS5 (full text search) built in. If you have Python, you already have a full text search engine. Using FTS5 directly is a bit awkward, so I wrote sqlitesearch as a convenient wrapper around it.

Install it:

```bash
uv add sqlitesearch
```

In [1]:
from ingest import load_faq_data

In [2]:
documents = load_faq_data()

In [3]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [ ]:
sqlite_index.search('').__len__()

0

In [15]:
sqlite_index.count()

85

Run this cell a few times while the other notebook is still ingesting.
You'll see the number growing as ingestion progresses. This is normal
database behavior: one process writes, another reads, both at the same
time. With minsearch this is impossible, because the index lives in one
process's memory and nobody else can reach it.

Try a search:

In [4]:
sqlite_index.search('how do I join the course', )[:2]

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'}]

In [16]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'I missed the first homework - can I still get a certificate?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

In [5]:
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

True

In [6]:
from rag_helper import RAGBase
assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [ ]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join after it started. If you want a certificate, make sure to submit your project while submissions are still open.'

## Comparing the two approaches

With minsearch (single process):

```text
Startup: fetch data -> parse -> index -> ready
Every restart: repeat all steps
```

With sqlitesearch (two processes):

```text
Ingestion (runs once): fetch data -> parse -> write to faq.db
Query (runs every time): open faq.db -> search -> ready
```

The RAG assistant then reads from it:

```mermaid
flowchart TD

    subgraph RAG["RAG ASSISTANT"]
        U([🙂 User])
        APP[Application]
        DOCS[[D1 ... D5]]
        PROMPT[Build Prompt<br/>Question + Context]
        LLM[LLM]
        ANSWER([Answer])

        U -->|Question| APP
        DOCS --> APP
        APP --> PROMPT
        PROMPT --> LLM
        LLM --> ANSWER
        ANSWER --> U
    end

    subgraph KB["KNOWLEDGE BASE"]
        DB[(DB)]
    end

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
```



For our FAQ dataset, both produce good results. The difference
matters more at scale with diverse document lengths.

## Choosing an approach

Pick the right tool for your data:

- `minsearch`: single process, in-memory only, re-indexes on every
  startup. Use when data is small and indexing is fast.
- `sqlitesearch`: separate ingestion and query, file-based (SQLite),
  opens existing index. Use when data is large or ingestion is slow.

Use minsearch when you can load and index the data on startup without
noticeable delay. Switch to a persistent backend when ingestion takes
too long or when you need the index to survive restarts.

For larger production systems, use the same pattern with a different
backend:

- Elasticsearch
- OpenSearch
- Qdrant (vector database)
- Weaviate (vector database)

The architecture stays the same: one process ingests, another queries.

## Cleaning up

When you're done, close the database connection:


In [17]:
sqlite_index.close()

Or just let Python clean it up when the notebook kernel shuts down.


## Fine-tuning vs RAG

Instead of retrieving documents at query time, you could fine-tune
the LLM on your data.

Fine-tuning means taking a model's weights and adjusting them for
your specific use case.

This works, but it has downsides:

- It requires special hardware (GPUs) and tools we don't cover in
  this course
- It's difficult to update when new data arrives - you don't want to
  retrain the model every time a new FAQ entry is added
- The LLM already has internal knowledge, but RAG gives it access to
  information it wasn't trained on

RAG is more flexible, cheaper, and works with any LLM. In practice,
fine-tuning is rarely needed. I've never personally hit a case that
required it. When I analyzed around 2,500 job descriptions for my AI
engineering field guide, few asked for it. So focus on RAG first, and
reach for fine-tuning only when you genuinely need it.

## Moving forward | Extra (Optional)

Try these next steps:

- Try different prompts and see how the answers change
- Add more data sources to the knowledge base
- Experiment with different LLM models (GPT-4o, Claude, Gemini, local
  models via Ollama)
- Try Elasticsearch as a search backend